# Streamlit Dashboard using MySQL Database

In [4]:
import streamlit as st
import pandas as pd
import seaborn as sns
import mysql.connector

In [5]:
# PAGE CONFIG
st.set_page_config(
    page_title="Air Tracker",
    layout="wide",
    page_icon="✈️"
)

2026-05-27 22:31:07.671 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [ ]:





# PAGE CONFIG


st.set_page_config(
    page_title="Airport Operations Dashboard",
    layout="wide",
    page_icon="✈️"
)

st.title("✈️ Airport Operations & Flight Analytics Dashboard")

# ---------------------------------------------------
# DATABASE CONNECTION
# ---------------------------------------------------

@st.cache_resource
def get_connection():

    conn = mysql.connector.connect(
        host="localhost",
        user="root",
        password="your_password",
        database="airport_db"
    )

    return conn


conn = get_connection()

# ---------------------------------------------------
# LOAD DATA FROM SQL
# ---------------------------------------------------

@st.cache_data
def load_airports():

    query = """
    SELECT
        icao_code,
        name,
        city,
        country,
        timezone
    FROM airport
    """

    return pd.read_sql(query, conn)


@st.cache_data
def load_flights():

    query = """
    SELECT
        f.flight_number,
        f.airline,
        f.origin_icao,
        f.destination_icao,
        f.scheduled_departure,
        f.actual_departure,
        f.scheduled_arrival,
        f.actual_arrival,
        f.status,

        origin.name AS origin_airport,
        destination.name AS destination_airport,

        aircraft.model AS aircraft_model

    FROM flight f

    LEFT JOIN airport origin
        ON f.origin_icao = origin.icao_code

    LEFT JOIN airport destination
        ON f.destination_icao = destination.icao_code

    LEFT JOIN aircraft
        ON f.aircraft_registration = aircraft.aircraft_code
    """

    df = pd.read_sql(query, conn)

    # Convert datetime columns
    datetime_cols = [
        "scheduled_departure",
        "actual_departure",
        "scheduled_arrival",
        "actual_arrival"
    ]

    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


airports_df = load_airports()
flights_df = load_flights()

# ---------------------------------------------------
# SIDEBAR FILTERS
# ---------------------------------------------------

st.sidebar.header("Filters")

selected_airport = st.sidebar.selectbox(
    "Select Airport",
    airports_df["icao_code"].unique()
)

selected_status = st.sidebar.multiselect(
    "Flight Status",
    flights_df["status"].dropna().unique(),
    default=flights_df["status"].dropna().unique()
)

selected_airline = st.sidebar.multiselect(
    "Airline",
    flights_df["airline"].dropna().unique(),
    default=flights_df["airline"].dropna().unique()
)

# ---------------------------------------------------
# FILTER DATA
# ---------------------------------------------------

filtered_df = flights_df[
    (
        (flights_df["origin_icao"] == selected_airport)
        |
        (flights_df["destination_icao"] == selected_airport)
    )
    &
    (flights_df["status"].isin(selected_status))
    &
    (flights_df["airline"].isin(selected_airline))
]

# ---------------------------------------------------
# AIRPORT EXPLORATION
# ---------------------------------------------------

st.header("🌍 Airport Exploration")

airport_info = airports_df[
    airports_df["icao_code"] == selected_airport
]

if not airport_info.empty:

    col1, col2, col3 = st.columns(3)

    with col1:
        st.metric(
            "Airport",
            airport_info.iloc[0]["name"]
        )

    with col2:
        st.metric(
            "City",
            airport_info.iloc[0]["city"]
        )

    with col3:
        st.metric(
            "Timezone",
            airport_info.iloc[0]["timezone"]
        )

    st.dataframe(airport_info)

# ---------------------------------------------------
# FLIGHT TREND ANALYSIS
# ---------------------------------------------------

st.header("📈 Flight Trend Analysis")

col1, col2 = st.columns(2)

# Flight Status Distribution
with col1:

    status_df = (
        filtered_df["status"]
        .value_counts()
        .reset_index()
    )

    status_df.columns = ["Status", "Count"]

    fig1 = px.pie(
        status_df,
        names="Status",
        values="Count",
        title="Flight Status Distribution"
    )

    st.plotly_chart(fig1, use_container_width=True)

# Airline Distribution
with col2:

    airline_df = (
        filtered_df["airline"]
        .value_counts()
        .reset_index()
    )

    airline_df.columns = ["Airline", "Flights"]

    fig2 = px.bar(
        airline_df,
        x="Airline",
        y="Flights",
        title="Flights by Airline"
    )

    st.plotly_chart(fig2, use_container_width=True)

# ---------------------------------------------------
# DEPARTURE TREND
# ---------------------------------------------------

st.subheader("🛫 Daily Departure Trend")

filtered_df["departure_date"] = (
    filtered_df["scheduled_departure"].dt.date
)

trend_df = (
    filtered_df.groupby("departure_date")
    .size()
    .reset_index(name="Flights")
)

fig3 = px.line(
    trend_df,
    x="departure_date",
    y="Flights",
    markers=True,
    title="Flights per Day"
)

st.plotly_chart(fig3, use_container_width=True)

# ---------------------------------------------------
# OPERATIONAL INSIGHTS
# ---------------------------------------------------

st.header("⚙️ Operational Insights")

# Delay Calculation
filtered_df["departure_delay"] = (
    (
        filtered_df["actual_departure"]
        -
        filtered_df["scheduled_departure"]
    ).dt.total_seconds() / 60
)

filtered_df["arrival_delay"] = (
    (
        filtered_df["actual_arrival"]
        -
        filtered_df["scheduled_arrival"]
    ).dt.total_seconds() / 60
)

col1, col2, col3 = st.columns(3)

with col1:

    avg_dep_delay = round(
        filtered_df["departure_delay"].mean(),
        2
    )

    st.metric(
        "Avg Departure Delay (mins)",
        avg_dep_delay
    )

with col2:

    avg_arr_delay = round(
        filtered_df["arrival_delay"].mean(),
        2
    )

    st.metric(
        "Avg Arrival Delay (mins)",
        avg_arr_delay
    )

with col3:

    cancelled_count = filtered_df[
        filtered_df["status"] == "Cancelled"
    ].shape[0]

    st.metric(
        "Cancelled Flights",
        cancelled_count
    )

# ---------------------------------------------------
# DELAY DISTRIBUTION
# ---------------------------------------------------

st.subheader("⏱ Departure Delay Distribution")

fig4 = px.histogram(
    filtered_df,
    x="departure_delay",
    nbins=30,
    title="Departure Delay Histogram"
)

st.plotly_chart(fig4, use_container_width=True)

# ---------------------------------------------------
# AIRLINE PERFORMANCE
# ---------------------------------------------------

st.header("🏆 Airline Performance")

performance_df = (
    filtered_df.groupby("airline")
    .agg(
        total_flights=("flight_number", "count"),
        avg_dep_delay=("departure_delay", "mean"),
        avg_arr_delay=("arrival_delay", "mean")
    )
    .reset_index()
)

performance_df["avg_dep_delay"] = (
    performance_df["avg_dep_delay"].round(2)
)

performance_df["avg_arr_delay"] = (
    performance_df["avg_arr_delay"].round(2)
)

st.dataframe(
    performance_df,
    use_container_width=True
)

# ---------------------------------------------------
# DECISION SUPPORT
# ---------------------------------------------------

st.header("🧠 Decision Support")

if not performance_df.empty:

    busiest_airline = (
        performance_df.sort_values(
            by="total_flights",
            ascending=False
        )
        .iloc[0]["airline"]
    )

    most_delayed_airline = (
        performance_df.sort_values(
            by="avg_dep_delay",
            ascending=False
        )
        .iloc[0]["airline"]
    )

    st.success(f"""
    ✔ Busiest Airline: {busiest_airline}

    ✔ Airline with Highest Delays: {most_delayed_airline}

    ✔ Airports can use these insights for:
       - Gate allocation
       - Staff scheduling
       - Delay reduction strategies
       - Traffic forecasting
    """)

# ---------------------------------------------------
# RAW DATA
# ---------------------------------------------------

with st.expander("View Flight Data"):
    st.dataframe(filtered_df)

# ---------------------------------------------------
# CLOSE CONNECTION
# ---------------------------------------------------

conn.close()